# 암종 분류 AI 해커톤 — 베이스라인 & EDA

**규정 준수 사항** (팀 협업 규정 v2)
- CV: `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` — 건드리지 않는다
- 평가: `f1_score(average="macro")`
- 모든 `fit`은 train에만. test에는 적용만.
- 실행 후 결과는 노션 【📋 실험·제출 기록】에 `test_0NN`으로 기록


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report

SEED = 42
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

from pathlib import Path

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs" / "baseline.yaml").exists():
            return path
    raise FileNotFoundError("저장소 루트를 찾지 못했습니다. JupyterLab을 레포 루트에서 실행하세요.")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data" / "raw"
train = pd.read_csv(f"{DATA}/train.csv")
test  = pd.read_csv(f"{DATA}/test.csv")
sub   = pd.read_csv(f"{DATA}/sample_submission.csv")

print(train.shape, test.shape, sub.shape)
train.iloc[:3, :6]

---
## 1. EDA — 첫날에 반드시 볼 것

### 1-1. 클래스 분포 (Macro F1의 출발점)

In [ ]:
y = train["SUBCLASS"]
vc = y.value_counts()

print(f"클래스 수: {y.nunique()}")
print(f"최소 {vc.min()}개({vc.idxmin()})  최대 {vc.max()}개({vc.idxmax()})  불균형비 {vc.max()/vc.min():.1f}배")
print(f"5개 미만 클래스: {(vc<5).sum()}개  →  StratifiedKFold(5) 사용에 문제 없음")
vc

> **해석** — 최소 클래스(DLBC)도 38개라 `StratifiedKFold(5)`가 폴드당 7~8개를 확보합니다. 경고 없이 그대로 씁니다.
>
> 다만 Macro F1은 26개 클래스의 F1을 **단순 평균**합니다. DLBC(38개)와 BRCA(786개)의 비중이 **똑같습니다**.
> 즉 소수 클래스 F1을 끌어올리는 것이 다수 클래스를 조금 더 맞히는 것보다 훨씬 이득입니다.

### 1-2. 변이 데이터의 형태

In [ ]:
gene_cols = list(train.columns[2:])
print(f"유전체 컬럼 {len(gene_cols)}개, 결측 {train[gene_cols].isna().sum().sum()}개")

# 값 형태: 'WT'(정상) 또는 변이 표기. 공백으로 구분된 다중 변이도 존재
print(train["TP53"].value_counts().head(5).to_dict())

mut = (train[gene_cols] != "WT")
rate = mut.mean()
print(f"\n유전자별 변이율: 중앙값 {rate.median():.4f}  최대 {rate.max():.4f} ({rate.idxmax()})")
print(f"변이율 0% 유전자(상수 컬럼): {(rate==0).sum()}개  →  즉시 제거 대상")
print(f"변이율 1% 미만 유전자: {(rate<0.01).sum()}개 ({(rate<0.01).mean()*100:.0f}%)  →  매우 희소")

rate.sort_values(ascending=False).head(15) * 100

> **해석** — TP53(28.5%), PIK3CA(11.1%)가 최상위. 암 유전체에서 알려진 순서와 일치합니다.
>
> 전체의 76%가 변이율 1% 미만입니다. **극도로 희소한 이진 행렬**이라고 보면 됩니다.

### 1-3. 환자별 변이 개수 (TMB) — 가장 강력한 단일 피처 후보

In [ ]:
n_mut = mut.sum(axis=1)
print(f"환자별 변이 개수: 중앙값 {n_mut.median():.0f}  평균 {n_mut.mean():.1f}  최대 {n_mut.max()}")

tmb = pd.DataFrame({"SUBCLASS": y, "n_mut": n_mut}).groupby("SUBCLASS")["n_mut"].median()
tmb.sort_values(ascending=False)

> **해석** — SKCM(흑색종) 85.5 vs THYM(흉선종) 2.0. **40배 차이**입니다.
> TMB(Tumor Mutational Burden)는 암종을 가르는 실제 임상 지표이고, 데이터에도 그대로 나타납니다.
>
> 이건 **한 행 안에서 세는 연산**이므로 Data Leakage가 아닙니다. 가장 먼저 넣어야 할 피처.

---
## 2. 전처리 — Leakage 없이

핵심 원칙: **행 내부 연산만 사용**하거나, **train에서 결정한 기준을 test에 적용**.

- `!= "WT"` 이진화 → 행 내부 연산, 안전
- 상수 컬럼 제거 → **train 기준**으로 컬럼을 고르고 test에 같은 컬럼을 적용, 안전
- 절대 금지: train+test를 합쳐 변이 빈도/희귀도를 계산하는 것

In [ ]:
def binarize(df, cols):
    """WT가 아니면 1. 행 내부 연산이므로 test에 그대로 적용해도 leakage 아님."""
    return (df[cols].values != "WT").astype(np.float32)

Xtr_raw = binarize(train, gene_cols)
Xte_raw = binarize(test,  gene_cols)

# 상수 컬럼 제거 — 기준은 train에서만 계산
keep = Xtr_raw.sum(axis=0) > 0
print(f"제거된 상수 컬럼 {(~keep).sum()}개 → 남은 컬럼 {keep.sum()}개")

def add_tmb(X):
    return np.hstack([X, np.log1p(X.sum(axis=1, keepdims=True))]).astype(np.float32)

X      = add_tmb(Xtr_raw[:, keep])
X_test = add_tmb(Xte_raw[:, keep])
print("X", X.shape, " X_test", X_test.shape)

---
## 3. 베이스라인 3종

아래 세 개는 이미 돌려본 결과입니다. **재현되는지 먼저 확인**하고 여기서 출발하세요.

| 모델 | CV Macro F1 |
|---|---|
| **LogisticRegression (balanced)** | **0.36296** |
| LGBM (class_weight=balanced) | 0.33806 |
| LGBM (기본) | 0.32608 |

> ⚠️ **GBDT가 지지 않는다는 법이 없습니다.** 행 6,201 대 컬럼 4,384의 희소 이진 데이터에서는
> 선형 모델이 트리보다 강한 경우가 흔합니다. "일단 LightGBM"으로 시작하지 마세요.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced", n_jobs=-1)
oof_lr = cross_val_predict(lr, X, y, cv=CV, n_jobs=1)
print(f"LogReg(balanced)  CV Macro F1 = {f1_score(y, oof_lr, average='macro'):.5f}")
np.save("oof_test_001.npy", oof_lr)

In [ ]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(objective="multiclass", n_estimators=300, learning_rate=0.1,
                     num_leaves=31, colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                     class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1)
oof_lgb = cross_val_predict(lgb, X, y, cv=CV, n_jobs=1)
print(f"LGBM(balanced)    CV Macro F1 = {f1_score(y, oof_lgb, average='macro'):.5f}")
np.save("oof_test_002.npy", oof_lgb)

### 클래스별로 뜯어보기 — 어디서 점수를 잃고 있나

In [ ]:
rep = pd.DataFrame(classification_report(y, oof_lr, output_dict=True, zero_division=0)).T
rep = rep.drop(index=["accuracy","macro avg","weighted avg"]).sort_values("f1-score")
rep[["precision","recall","f1-score","support"]].round(3)

> **여기가 전략의 출발점입니다.** F1이 낮은 하위 클래스 몇 개를 개선하는 것이
> Macro F1에 미치는 영향이 가장 큽니다. 상위 클래스를 더 짜내지 마세요.

---
## 4. 제출 파일 생성

In [ ]:
model = LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced", n_jobs=-1)
model.fit(X, y)                    # fit은 train에서 단 한 번
pred = model.predict(X_test)       # test에는 적용만

sub["SUBCLASS"] = pred
sub.to_csv("submission_test_001.csv", index=False)
print(sub["SUBCLASS"].value_counts().head())
sub.head()

---
## 5. 제출 전 셀프 체크 (규정 5)

- [ ] 코드 어디에도 `test`를 대상으로 한 `fit` / `fit_transform`이 없다
- [ ] `pd.concat([train, test])` 후 일괄 전처리한 구간이 없다
- [ ] 유전자별 변이 빈도·희귀도 같은 집계 피처를 **train만으로** 계산했다
- [ ] 피처 선택 기준을 train에서만 계산했다
- [ ] 노트북을 위에서부터 다시 실행했을 때 같은 CV 점수가 나온다

체크 후 노션 【📋 실험·제출 기록】에서 `Leakage 점검` → **'확인 완료'**로 변경.

---
## 6. 다음에 시도해볼 것 (차원별 담당자에게)

**① 변이 인코딩** — 지금은 WT 여부 이진화뿐입니다.
- 유전자당 변이 **개수**(다중 변이 표기가 10% 존재)
- 변이 타입 분해 — missense / nonsense / silent 구분 (`R175H` vs `R175R`)
- 유전자군·경로 단위 집계

**② 차원 축소** — 4,231개는 과합니다.
- 변이율 하한 필터 (예: 1% 이상만 → 약 900개)
- FI 기반 pruning, TruncatedSVD

**③ 클래스 불균형** — `balanced`는 시작일 뿐
- 클래스별 임계값 조정, 소수 클래스 오버샘플링

**④ 모델 다양성** — 선형이 이기고 있다는 사실에 주목
- LinearSVC, ComplementNB, kNN(자카드 거리)
- **앙상블은 확률 평균 후 argmax.** Rank Average는 Macro F1에서 못 씁니다
